# 02 · EfficientNet-B0 and training interventions
EfficientNet-B0 light-only (5 seeds), +FST IV–V training (3 seeds) and size-matched +FST IV–V training (3 seeds), plus per-type, source-site and subtype analyses for both architectures.

**Kaggle inputs:** the two datasets from notebook 01, and notebook 01's saved output. GPU (T4), internet on.
**Outputs:** all 16 original-image probability files (`probs_seed*.csv`), used by notebooks 03 and 04.

## 1. Verify the original images

In [ ]:
%%writefile verify_orig.py
"""Check every attached original-image dataset, pick the best, and stop if FST VI coverage is poor."""
import glob, hashlib, os, random
import numpy as np, pandas as pd
from PIL import Image

IN = os.environ.get("INPUT_ROOT", "/kaggle/input")
pd.set_option("display.width", 200)
csvs = [c for c in sorted(glob.glob(f"{IN}/**/*.csv", recursive=True))
        if {"md5hash", "fitzpatrick_scale"} <= set(pd.read_csv(c, nrows=1).columns)]
csvs.sort(key=lambda c: "original" in c.lower())          # prefer a CSV outside the image datasets
if not csvs:
    raise FileNotFoundError("Fitzpatrick17k CSV not found: attach the Nazmus Resan dataset (for its CSV)")
meta = pd.read_csv(csvs[0])
v = meta[meta.fitzpatrick_scale.between(1, 6)].copy()
print(f"[orig] csv: {csvs[0]}")

if os.environ.get("IMG_ROOT"):
    roots = [os.environ["IMG_ROOT"]]
else:
    cand = sorted({d.rstrip("/") for d in glob.glob(f"{IN}/**/", recursive=True) if "original" in d.lower()}, key=len)
    roots = [d for d in cand if not any(d != r and d.startswith(r + "/") for r in cand)]
if not roots:
    raise FileNotFoundError("No original-image dataset found: attach fitzpatrick17k_original and/or asosenge's Fitzpatrick17K")


def check(root):
    files = {}
    for ext in ("jpg", "jpeg", "png", "JPG", "JPEG", "PNG"):
        for p in glob.glob(f"{root}/**/*.{ext}", recursive=True):
            files[os.path.splitext(os.path.basename(p))[0]] = p
    x = v.assign(found=v.md5hash.isin(files))
    cov = x.groupby("fitzpatrick_scale").agg(rows=("found", "size"), found=("found", "sum"))
    mal = x[x.three_partition_label == "malignant"]
    cov["malignant_rows"] = mal.groupby("fitzpatrick_scale").size()
    cov["malignant_found"] = mal[mal.found].groupby("fitzpatrick_scale").size()
    cov = cov.fillna(0).astype(int)
    cov["coverage"] = (cov.found / cov.rows).round(4)
    samp = random.Random(0).sample(list(files.items()), min(300, len(files))) if files else []
    match = sum(hashlib.md5(open(p, "rb").read()).hexdigest() == h for h, p in samp)
    black = 0
    for h, p in samp:
        a = np.asarray(Image.open(p).convert("RGB").resize((64, 64))).astype(int)
        c = np.concatenate([a[:4, :4], a[:4, -4:], a[-4:, :4], a[-4:, -4:]]).reshape(-1, 3)
        black += int(np.median(c.max(axis=1)) < 30)
    print("\n" + "=" * 70 + f"\n{root}\n" + "=" * 70)
    print(f"images: {len(files)}")
    print(cov.to_string())
    print(f"file fingerprint matches name: {match}/{len(samp)}   (near {len(samp)} = untouched originals)")
    print(f"black-corner images in sample: {black}/{len(samp)}   (originals ~6% from dark photo borders; masked copy ~54%)")
    return dict(root=root, cov6=cov.coverage.get(6, 0), covmin=cov.coverage.min(),
                found=int(cov.found.sum()), match=match / max(len(samp), 1), black=black / max(len(samp), 1))


R = [check(r) for r in roots]
best = max(R, key=lambda r: (r["cov6"] >= .95, r["match"] >= .95, r["cov6"], r["match"], r["found"]))
print("\n" + "=" * 70)
print(pd.DataFrame(R).round(3).to_string(index=False))
print(f"\nchosen: {best['root']}")
if best["cov6"] < 0.95 or best["covmin"] < 0.90 or best["match"] < 0.95:
    raise RuntimeError("No attached dataset is clean and complete enough - stopping before training. Check the attached datasets.")
os.environ["IMG_ROOT"] = best["root"]
os.environ["CSV_PATH"] = csvs[0]
print("OK - IMG_ROOT set. Training cells will use only these images.")

In [ ]:
%run verify_orig.py

## 2. Copy notebook 01 results (original-image runs only) into /kaggle/working

In [ ]:
import glob, json, os, shutil
W = "/kaggle/working"
n = skipped = 0
for jf in glob.glob("/kaggle/input/**/results_seed*.json", recursive=True):
    try:
        cfg = json.load(open(jf)).get("config", {})
    except Exception:
        continue
    if "smoke" in jf:
        continue
    if "original" not in str(cfg.get("img_root", "")).lower():
        skipped += 1          # produced from the masked images: never mix these in
        continue
    pf = jf.replace("results_", "probs_").replace(".json", ".csv")
    for src in (jf, pf):
        dst = os.path.join(W, os.path.basename(src))
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copy(src, dst); n += 1
print(f"copied {n} files; skipped {skipped} results made from masked images")
have = sorted(f for f in os.listdir(W) if f.startswith("probs_seed"))
print(have)
missing = [s for s in range(5) if f"probs_seed{s}.csv" not in have]
if missing:
    raise FileNotFoundError(f"probs_seed{missing}.csv missing: attach notebook 01 (the original-image run)")

## 3. Scripts

In [ ]:
%%writefile skin_tone_gap.py
"""
Skin-tone generalisation gap in dermatology classifiers.
Train on light skin (FST I-III), evaluate on light AND dark (FST IV-VI).

USAGE
-----
Smoke test first (2 min, proves the pipeline runs):
    SMOKE=1 python skin_tone_gap.py

Full run, one seed:
    SEED=0 python skin_tone_gap.py

Mitigation run (oversamples dark skin in training):
    SEED=0 MITIGATE=1 python skin_tone_gap.py

In a Kaggle notebook cell:
    import os; os.environ["SMOKE"] = "1"
    %run skin_tone_gap.py

OUTPUTS (to OUT_DIR)
--------------------
    results_seed{N}.json      per-group metrics
    probs_seed{N}.csv         raw per-image probabilities  <-- keep these!
    model_seed{N}.pt          checkpoint for Grad-CAM later

The probs CSV is the important artefact: it lets you compute NEW metrics
tomorrow (ECE, bootstrapped CIs, subgroup splits) WITHOUT retraining.
"""

import os
import glob
import json
import random
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import roc_auc_score, confusion_matrix
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------
SEED      = int(os.environ.get("SEED", 0))
SMOKE     = os.environ.get("SMOKE", "0") == "1"
MITIGATE  = os.environ.get("MITIGATE", "0") == "1"
EVAL_ONLY = os.environ.get("EVAL_ONLY", "0") == "1"   # load checkpoint, skip training
RESUME    = os.environ.get("RESUME", "0") == "1"      # continue from checkpoint

# MIT_TARGET: which FST levels the mitigation oversamples.
#   "dark" = all of FST 4-6 (the naive version)
#   "6"    = FST 6 only -- use this if the cliff is at FST 6, not across 4-6
MIT_TARGET = os.environ.get("MIT_TARGET", "dark")

# HOLDOUT_FST: train on light + these FST levels, test on the rest.
#   e.g. HOLDOUT_FST="4,5" trains on light+FST4+FST5 and tests on FST6.
#   Isolates "never saw dark skin" from "FST 6 is intrinsically harder".
HOLDOUT_FST = os.environ.get("HOLDOUT_FST", "")

# SIZE_MATCH=1 with HOLDOUT_FST: drop light training images so the training set
#   stays the same size as the light-only baseline. Separates "saw darker skin"
#   from "simply had more training data".
SIZE_MATCH = os.environ.get("SIZE_MATCH", "0") == "1"

# ARCH: "resnet50" (default) or "effb0" (EfficientNet-B0). Second backbone checks
#   whether the result belongs to the data or to one architecture.
ARCH = os.environ.get("ARCH", "resnet50")
PRETRAINED = os.environ.get("PRETRAINED", "1") == "1"   # 0 only for offline tests

IMG_SIZE   = 224
BATCH_SIZE = 32
EPOCHS     = 2 if SMOKE else int(os.environ.get("EPOCHS", 10))
LR         = 3e-4
N_SMOKE    = 200          # images used in smoke mode

DATA_ROOT = os.environ.get("DATA_ROOT", "/kaggle/input")
IMG_ROOT  = os.environ.get("IMG_ROOT", "") or DATA_ROOT   # where images are looked up (CSV still from DATA_ROOT)
OUT_DIR   = os.environ.get("OUT_DIR", "/kaggle/working")
os.makedirs(OUT_DIR, exist_ok=True)

DEVICE = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available()
          else "cpu")

tag = (f"seed{SEED}"
       + (f"_{ARCH}" if ARCH != "resnet50" else "")
       + (f"_mitigate{MIT_TARGET}" if MITIGATE else "")
       + (f"_hold{HOLDOUT_FST.replace(',', '')}" if HOLDOUT_FST else "")
       + ("_sm" if (HOLDOUT_FST and SIZE_MATCH) else "")
       + ("_smoke" if SMOKE else ""))


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


# ----------------------------------------------------------------------------
# 1. FIND THE DATA
#    Kaggle mirrors differ. We sniff for the CSV and the image folder.
# ----------------------------------------------------------------------------
def find_csv():
    if os.environ.get("CSV_PATH"):
        return os.environ["CSV_PATH"]
    cands = glob.glob(f"{DATA_ROOT}/**/*.csv", recursive=True)
    if not cands:
        raise FileNotFoundError(f"No CSV under {DATA_ROOT}. Is the dataset attached?")
    # prefer one that looks like the fitzpatrick index
    for c in cands:
        if "fitz" in os.path.basename(c).lower():
            return c
    return max(cands, key=os.path.getsize)


def build_image_index():
    """Map stem -> full path for every image under IMG_ROOT."""
    idx = {}
    for ext in ("jpg", "jpeg", "png", "JPG", "JPEG", "PNG"):
        for p in glob.glob(f"{IMG_ROOT}/**/*.{ext}", recursive=True):
            idx[os.path.splitext(os.path.basename(p))[0]] = p
    if not idx:
        raise FileNotFoundError(f"No images found under {IMG_ROOT}")
    print(f"[data] image root: {IMG_ROOT}  ({len(idx)} images indexed)")
    return idx


def pick_col(df, options, what):
    low = {c.lower(): c for c in df.columns}
    for o in options:
        if o in low:
            return low[o]
    raise KeyError(
        f"Could not find a {what} column. Looked for {options}.\n"
        f"Columns present: {list(df.columns)}\n"
        f"--> Edit pick_col() with the right name."
    )


print(f"[setup] device={DEVICE}  seed={SEED}  arch={ARCH}  smoke={SMOKE}  "
      f"mitigate={MITIGATE}  holdout={HOLDOUT_FST or '-'}  size_match={SIZE_MATCH}")

csv_path = find_csv()
df = pd.read_csv(csv_path)
print(f"[data] csv: {csv_path}  rows={len(df)}")
print(f"[data] columns: {list(df.columns)}")

FST_COL   = pick_col(df, ["fitzpatrick_scale", "fitzpatrick", "fitzpatrick_centaur",
                          "fst", "skin_type"], "Fitzpatrick")
LABEL_COL = pick_col(df, ["three_partition_label", "label", "diagnosis",
                          "nine_partition_label", "target"], "label")
ID_COL    = pick_col(df, ["md5hash", "image_id", "filename", "image", "id",
                          "url_alphanum"], "image id")

print(f"[data] using  fst={FST_COL}  label={LABEL_COL}  id={ID_COL}")

# ----------------------------------------------------------------------------
# 2. BUILD LABELS AND GROUPS
# ----------------------------------------------------------------------------
df[FST_COL] = pd.to_numeric(df[FST_COL], errors="coerce")
df = df[df[FST_COL].between(1, 6)].copy()          # drop -1 / unknown

df["group"] = np.where(df[FST_COL] <= 3, "light", "dark")

# Binary target: malignant vs not. Adjust the keyword if your label space differs.
lab = df[LABEL_COL].astype(str).str.lower()
df["y"] = lab.str.contains("malignant").astype(int)

if df["y"].nunique() < 2:
    raise ValueError(
        f"Binary target collapsed. Unique {LABEL_COL} values: "
        f"{df[LABEL_COL].unique()[:20]}\n--> Fix the 'malignant' rule above."
    )

img_index = build_image_index()
df["path"] = df[ID_COL].astype(str).map(img_index)
missing = df["path"].isna().sum()
df = df.dropna(subset=["path"]).reset_index(drop=True)
print(f"[data] resolved {len(df)} images ({missing} unresolved / URL-rotted)")

print("\n[data] group x class counts")
print(pd.crosstab(df["group"], df["y"]))

# ----------------------------------------------------------------------------
# 3. SPLIT  -- train on LIGHT only; test on both groups
# ----------------------------------------------------------------------------
set_seed(SEED)
light = df[df.group == "light"].sample(frac=1, random_state=SEED).reset_index(drop=True)
dark  = df[df.group == "dark"].reset_index(drop=True)

n_test = int(0.20 * len(light))
test_light  = light.iloc[:n_test]
train_light = light.iloc[n_test:]

if HOLDOUT_FST:
    # train on light + the named FST levels; test on the FST levels NOT named
    keep = [int(v) for v in HOLDOUT_FST.split(",")]
    train_extra = dark[dark[FST_COL].isin(keep)]
    test_dark   = dark[~dark[FST_COL].isin(keep)]
    light_part  = train_light
    if SIZE_MATCH:
        n_light = len(train_light) - len(train_extra)
        if n_light <= 0:
            raise ValueError("SIZE_MATCH: more extra images than the baseline training set")
        light_part = train_light.sample(n_light, random_state=SEED)
        print(f"[holdout] size-matched: {n_light} light + {len(train_extra)} FST {keep} "
              f"= {n_light + len(train_extra)} (baseline {len(train_light)})")
    train = pd.concat([light_part, train_extra], ignore_index=True)
    print(f"[holdout] training also on FST {keep}; testing on FST "
          f"{sorted(test_dark[FST_COL].unique().astype(int))}")
elif MITIGATE:
    # hold out half the targeted data for testing, put the rest in training
    pool = dark if MIT_TARGET == "dark" else dark[dark[FST_COL] == int(MIT_TARGET)]
    rest = dark.drop(pool.index)
    pool = pool.sample(frac=1, random_state=SEED)
    cut = len(pool) // 2
    test_dark   = pd.concat([pool.iloc[:cut], rest], ignore_index=True)
    train_extra = pool.iloc[cut:]
    train = pd.concat([train_light, train_extra], ignore_index=True)
    print(f"[mitigate] oversampling target='{MIT_TARGET}', "
          f"{len(train_extra)} images added to training")
else:
    test_dark = dark
    train = train_light

if SMOKE:
    train     = train.sample(min(N_SMOKE, len(train)), random_state=SEED)
    test_light = test_light.sample(min(100, len(test_light)), random_state=SEED)
    test_dark  = test_dark.sample(min(100, len(test_dark)),  random_state=SEED)

print(f"\n[split] train={len(train)}  test_light={len(test_light)}  test_dark={len(test_dark)}")
print(f"[split] test_dark positives={int(test_dark.y.sum())}")
if len(test_dark) < 200 and not SMOKE:
    print("[WARN] dark test set is small -- expect wide CIs. "
          "Consider bootstrapping rather than relying on seed variance alone.")

# ----------------------------------------------------------------------------
# 4. DATA PIPELINE
# ----------------------------------------------------------------------------
NORM = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(), NORM,
])
# NOTE: deliberately NO colour jitter -- it would confound the skin-tone signal.

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(), NORM,
])


class SkinDS(Dataset):
    def __init__(self, frame, tf):
        self.df, self.tf = frame.reset_index(drop=True), tf

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(r["path"]).convert("RGB")
        return self.tf(img), int(r["y"]), i


NW = int(os.environ.get("NUM_WORKERS", 4 if DEVICE == "cuda" else 0))


def loader(frame, tf, shuffle=False, sampler=None):
    return DataLoader(SkinDS(frame, tf), batch_size=BATCH_SIZE,
                      shuffle=shuffle and sampler is None,
                      sampler=sampler, num_workers=NW, pin_memory=(DEVICE == "cuda"))


sampler = None
if MITIGATE:
    # oversample the targeted images so the model sees them near parity
    if MIT_TARGET == "dark":
        is_target = (train["group"].values == "dark")
    else:
        is_target = (train[FST_COL].values == int(MIT_TARGET))
    n_t = max(is_target.sum(), 1)
    n_o = max((~is_target).sum(), 1)
    w = np.where(is_target, len(train) / n_t, len(train) / n_o)
    sampler = WeightedRandomSampler(torch.DoubleTensor(w), len(train), replacement=True)
    print(f"[mitigate] sampler weights: {n_t} target vs {n_o} other")

train_dl = loader(train, train_tf, shuffle=True, sampler=sampler)
light_dl = loader(test_light, eval_tf)
dark_dl  = loader(test_dark,  eval_tf)

# ----------------------------------------------------------------------------
# 5. MODEL
# ----------------------------------------------------------------------------
if ARCH == "resnet50":
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2 if PRETRAINED else None)
    model.fc = nn.Linear(model.fc.in_features, 2)
elif ARCH == "effb0":
    model = models.efficientnet_b0(
        weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1 if PRETRAINED else None)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
else:
    raise ValueError(f"Unknown ARCH '{ARCH}' (use resnet50 or effb0)")
print(f"[model] {ARCH}  pretrained={PRETRAINED}")
model = model.to(DEVICE)

# class-weighted loss: malignant is the minority and the one that matters
counts = train["y"].value_counts()
cw = torch.tensor([len(train) / (2 * counts.get(0, 1)),
                   len(train) / (2 * counts.get(1, 1))],
                  dtype=torch.float32, device=DEVICE)
crit = nn.CrossEntropyLoss(weight=cw)
opt  = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

# ----------------------------------------------------------------------------
# 6. TRAIN
# ----------------------------------------------------------------------------
CKPT = f"{OUT_DIR}/model_{tag}.pt"

if EVAL_ONLY or RESUME:
    if not os.path.exists(CKPT):
        raise FileNotFoundError(
            f"No checkpoint at {CKPT}. The session was wiped -- retrain from scratch."
        )
    model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
    print(f"[ckpt] loaded {CKPT}")

if EVAL_ONLY:
    print("[eval] skipping training, going straight to evaluation")
else:
    for ep in range(EPOCHS):
        model.train()
        tot = 0.0
        for x, y, _ in train_dl:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
                loss = crit(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            tot += loss.item() * x.size(0)
        sched.step()
        print(f"[train] epoch {ep+1}/{EPOCHS}  loss={tot/len(train_dl.dataset):.4f}",
              flush=True)   # flush: Kaggle buffers stdout and looks frozen otherwise
        # checkpoint EVERY epoch -- Kaggle sessions die without warning
        torch.save(model.state_dict(), CKPT)

# ----------------------------------------------------------------------------
# 7. EVALUATE
# ----------------------------------------------------------------------------
@torch.no_grad()
def predict(dl):
    model.eval()
    P, Y, I = [], [], []
    for x, y, i in dl:
        p = torch.softmax(model(x.to(DEVICE)), 1)[:, 1]
        P.append(p.cpu().numpy()); Y.append(y.numpy()); I.append(i.numpy())
    return np.concatenate(P), np.concatenate(Y), np.concatenate(I)


def ece(probs, y, bins=10):
    """Expected calibration error -- keep this even if the accuracy gap is small."""
    conf = np.where(probs >= 0.5, probs, 1 - probs)
    correct = ((probs >= 0.5).astype(int) == y).astype(float)
    edges, e = np.linspace(0, 1, bins + 1), 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum():
            e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(e)


def metrics(probs, y):
    pred = (probs >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "n": int(len(y)),
        "n_pos": int(y.sum()),
        "sensitivity": float(tp / (tp + fn)) if (tp + fn) else None,
        "specificity": float(tn / (tn + fp)) if (tn + fp) else None,
        "auc": float(roc_auc_score(y, probs)) if len(np.unique(y)) > 1 else None,
        "ece": ece(probs, y),
        "accuracy": float((pred == y).mean()),
    }


res, rows = {}, []
for name, dl, frame in [("light", light_dl, test_light), ("dark", dark_dl, test_dark)]:
    p, y, i = predict(dl)
    res[name] = metrics(p, y)
    sub = frame.reset_index(drop=True).iloc[i]
    rows.append(pd.DataFrame({
        "group": name, "prob": p, "y": y,
        "path": sub["path"].values, "fst": sub[FST_COL].values,
        # carry the diagnosis through so the disease-mix confound can be
        # checked later WITHOUT retraining
        "condition": sub[LABEL_COL].values,
    }))

if res["light"]["sensitivity"] is not None and res["dark"]["sensitivity"] is not None:
    res["gap_sensitivity"] = res["light"]["sensitivity"] - res["dark"]["sensitivity"]
if res["light"]["auc"] is not None and res["dark"]["auc"] is not None:
    res["gap_auc"] = res["light"]["auc"] - res["dark"]["auc"]
res["gap_ece"] = res["dark"]["ece"] - res["light"]["ece"]
res["config"] = {"seed": SEED, "smoke": SMOKE, "mitigate": MITIGATE,
                 "mit_target": MIT_TARGET if MITIGATE else None,
                 "holdout_fst": HOLDOUT_FST or None, "size_match": SIZE_MATCH,
                 "arch": ARCH, "epochs": EPOCHS, "n_train": len(train), "img_root": IMG_ROOT}

pd.concat(rows).to_csv(f"{OUT_DIR}/probs_{tag}.csv", index=False)
with open(f"{OUT_DIR}/results_{tag}.json", "w") as f:
    json.dump(res, f, indent=2)

print("\n" + "=" * 60)
print(json.dumps(res, indent=2))
print("=" * 60)
print(f"saved -> {OUT_DIR}/results_{tag}.json  /  probs_{tag}.csv  /  model_{tag}.pt")

In [ ]:
%%writefile analyze_fst.py
"""
Per-Fitzpatrick-level analysis across ALL seeds, plus the paired checks.
CPU only, about a minute. Reads the probs_*.csv files written by skin_tone_gap.py.

In a Kaggle cell:   %run analyze_fst.py
Needs:     probs_seed{0-4}.csv                      (main runs)
Optional:  probs_seed{0-2}_mitigate6.csv            (mitigation)
           probs_seed{0-2}_hold45.csv               (holdout)
           the Fitzpatrick17k CSV under /kaggle/input (subtype check)
"""
import glob
import os
import re

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

OUT  = os.environ.get("OUT_DIR", "/kaggle/working")
DATA = os.environ.get("DATA_ROOT", "/kaggle/input")
NB   = int(os.environ.get("NBOOT", 1000))
rng  = np.random.default_rng(0)
pd.set_option("display.width", 200, "display.precision", 3)


def sens(p, y):
    m = y == 1
    return float((p[m] >= 0.5).mean()) if m.any() else np.nan


def spec(p, y):
    m = y == 0
    return float((p[m] < 0.5).mean()) if m.any() else np.nan


def auc(p, y):
    return float(roc_auc_score(y, p)) if len(np.unique(y)) > 1 else np.nan


def summarize(g):
    p, y = g["prob"].values, g["y"].values
    return dict(n=len(g), pos=int(y.sum()), sens=sens(p, y), spec=spec(p, y), auc=auc(p, y))


def read_seed_files(suffix):
    found = {}
    for f in glob.glob(f"{OUT}/probs_seed*{suffix}.csv"):
        m = re.fullmatch(rf"probs_seed(\d+){re.escape(suffix)}\.csv", os.path.basename(f))
        if m:
            found[int(m.group(1))] = pd.read_csv(f)
    return dict(sorted(found.items()))


main = read_seed_files("")
mit  = read_seed_files("_mitigate6")
hold = read_seed_files("_hold45")
if not main:
    raise SystemExit(f"No probs_seed*.csv in {OUT}")
print(f"main seeds: {list(main)}   mitigate6 seeds: {list(mit)}   hold45 seeds: {list(hold)}")

# ---------------------------------------------------------------------------
print("\n" + "=" * 74)
print("1. PER-FST RESULTS ACROSS SEEDS (mean, SD over seeds; threshold 0.5)")
print("=" * 74)
rows = []
for s, d in main.items():
    for lvl, g in d.groupby("fst"):
        rows.append(dict(seed=s, fst=int(lvl), **summarize(g)))
T = pd.DataFrame(rows)
A = T.groupby("fst").agg(
    n=("n", "mean"), pos=("pos", "mean"),
    sens=("sens", "mean"), sens_sd=("sens", "std"),
    spec=("spec", "mean"), spec_sd=("spec", "std"),
    auc=("auc", "mean"), auc_sd=("auc", "std"))
print(A.round(3).to_string())
print("\nsensitivity by seed")
print(T.pivot(index="fst", columns="seed", values="sens").round(3).to_string())
print("\nAUC by seed")
print(T.pivot(index="fst", columns="seed", values="auc").round(3).to_string())

# ---------------------------------------------------------------------------
print("\n" + "=" * 74)
print("2. PRE-SPECIFIED CONTRASTS (bootstrap over test images, per seed)")
print("=" * 74)


def boot_gap(L, D, stat):
    pl, yl = L["prob"].values, L["y"].values
    pdk, yd = D["prob"].values, D["y"].values
    point = stat(pl, yl) - stat(pdk, yd)
    gaps = np.empty(NB)
    for b in range(NB):
        i = rng.integers(0, len(yl), len(yl))
        j = rng.integers(0, len(yd), len(yd))
        gaps[b] = stat(pl[i], yl[i]) - stat(pdk[j], yd[j])
    return point, np.nanpercentile(gaps, 2.5), np.nanpercentile(gaps, 97.5)


CONTRASTS = [("FST 1-3 minus FST 6",   [1, 2, 3], [6]),
             ("FST 1-3 minus FST 4-5", [1, 2, 3], [4, 5]),
             ("FST 4-5 minus FST 6",   [4, 5],    [6])]
for name, lo, hi in CONTRASTS:
    for metric, stat in (("sensitivity", sens), ("AUC", auc)):
        res = [(s, *boot_gap(d[d.fst.isin(lo)], d[d.fst.isin(hi)], stat))
               for s, d in main.items()]
        gaps = np.array([r[1] for r in res])
        sig = sum(1 for r in res if r[2] > 0 or r[3] < 0)
        print(f"{name:24s} {metric:12s} mean gap {gaps.mean():+.3f}  "
              f"per-seed [{gaps.min():+.3f}, {gaps.max():+.3f}]  "
              f"CI excludes 0 in {sig}/{len(res)} seeds")

# ---------------------------------------------------------------------------
print("\n" + "=" * 74)
print("3. MITIGATION, PAIRED ON THE SAME IMAGES")
print("   (baseline vs oversample-FST6; FST 6 images = the held-out half only)")
print("=" * 74)
rows = []
for s, m in mit.items():
    if s not in main:
        continue
    b = main[s]
    m6 = m[m.fst == 6]
    b6 = b[(b.fst == 6) & b.path.isin(m6.path)]
    m6 = m6[m6.path.isin(b6.path)]
    for tag, bb, mm in (("FST6 (same imgs)", b6, m6),
                        ("light test", b[b.group == "light"], m[m.group == "light"])):
        rows.append(dict(seed=s, subset=tag, model="baseline", **summarize(bb)))
        rows.append(dict(seed=s, subset=tag, model="oversample6", **summarize(mm)))
if rows:
    M = pd.DataFrame(rows)
    print(M.round(3).to_string(index=False))
    print("\nmean over seeds")
    print(M.groupby(["subset", "model"])[["sens", "spec", "auc"]].mean().round(3).to_string())
else:
    print("no mitigation files found")

# ---------------------------------------------------------------------------
print("\n" + "=" * 74)
print("4. HOLDOUT (train on light + FST 4,5) vs BASELINE, ON FST 6")
print("   NB: holdout also adds ~4,300 training images; tone exposure and data")
print("   volume are not separated by this comparison.")
print("=" * 74)
rows = []
for s, h in hold.items():
    if s not in main:
        continue
    b = main[s]
    rows.append(dict(seed=s, subset="FST6", model="baseline", **summarize(b[b.fst == 6])))
    rows.append(dict(seed=s, subset="FST6", model="train+FST4,5", **summarize(h[h.fst == 6])))
    rows.append(dict(seed=s, subset="light test", model="baseline", **summarize(b[b.group == "light"])))
    rows.append(dict(seed=s, subset="light test", model="train+FST4,5", **summarize(h[h.group == "light"])))
if rows:
    H = pd.DataFrame(rows)
    print(H.round(3).to_string(index=False))
    print("\nmean over seeds")
    print(H.groupby(["subset", "model"])[["sens", "spec", "auc"]].mean().round(3).to_string())
else:
    print("no holdout files found")

# ---------------------------------------------------------------------------
print("\n" + "=" * 74)
print("5. MALIGNANT SUBTYPE MIX (is FST 6 a different mix of cancers?)")
print("=" * 74)
csvs = [c for c in glob.glob(f"{DATA}/**/*.csv", recursive=True)
        if "fitz" in os.path.basename(c).lower()]
if os.environ.get("CSV_PATH"):
    csvs = [os.environ["CSV_PATH"]]
if not csvs:
    print("Fitzpatrick CSV not found under", DATA, "- skipped")
else:
    meta = (pd.read_csv(csvs[0])[["md5hash", "label"]]
            .rename(columns={"label": "dx"}).drop_duplicates("md5hash"))

    def with_dx(d):
        d = d.copy()
        d["md5hash"] = d["path"].map(lambda p: os.path.splitext(os.path.basename(p))[0])
        return d.merge(meta, on="md5hash", how="left")

    d0 = with_dx(main[min(main)])
    d0 = d0[d0.y == 1].copy()
    d0["grp"] = np.where(d0.fst <= 3, "FST1-3", np.where(d0.fst <= 5, "FST4-5", "FST6"))
    ct = pd.crosstab(d0.dx, d0.grp)
    share = (ct / ct.sum() * 100).round(1).add_suffix("_%")
    comp = ct.join(share).sort_values("FST6", ascending=False)
    print("malignant cases by diagnosis (counts, and % of that group's malignant cases)")
    print(comp.head(10).to_string())

    f6 = pd.concat([d.assign(seed=s) for s, d in main.items()])
    f6 = with_dx(f6[(f6.fst == 6) & (f6.y == 1)])
    f6["hit"] = f6.prob >= 0.5
    r = (f6.groupby("dx").agg(n=("path", "nunique"), sens=("hit", "mean"))
         .sort_values("n", ascending=False))
    print("\nFST 6 malignant cases: sensitivity by diagnosis (mean over seeds)")
    print(r.head(10).round(3).to_string())
    print("-> if FST 6 cancers are dominated by subtypes that are rare in the")
    print("   light-skin training set, the cliff is partly a subtype effect.")

In [ ]:
%%writefile analyze_subtype.py
"""
Subtype-stratified check: is the FST 6 cliff a skin-tone effect, a cancer-subtype
effect, or both? CPU only, seconds.

In a Kaggle cell (same setup as analyze_fst.py):
    os.environ["OUT_DIR"] = "<folder holding probs_*.csv>"
    %run analyze_subtype.py
Needs the Fitzpatrick17k CSV under /kaggle/input (for the diagnosis column).

All counts are small (FST 6 has 61 malignant images in total). Read the output as
descriptive, not as a hypothesis test.
"""
import glob
import os
import re

import numpy as np
import pandas as pd

OUT  = os.environ.get("OUT_DIR", "/kaggle/working")
DATA = os.environ.get("DATA_ROOT", "/kaggle/input")
pd.set_option("display.width", 200, "display.precision", 3)

main = {}
for f in glob.glob(f"{OUT}/probs_seed*.csv"):
    m = re.fullmatch(r"probs_seed(\d+)\.csv", os.path.basename(f))
    if m:
        main[int(m.group(1))] = pd.read_csv(f)
main = dict(sorted(main.items()))
if not main:
    raise SystemExit(f"No probs_seed*.csv in {OUT}")

csvs = [c for c in glob.glob(f"{DATA}/**/*.csv", recursive=True)
        if "fitz" in os.path.basename(c).lower()]
if os.environ.get("CSV_PATH"):
    csvs = [os.environ["CSV_PATH"]]
if not csvs:
    raise SystemExit(f"Fitzpatrick17k CSV not found under {DATA}")
meta = (pd.read_csv(csvs[0])[["md5hash", "label"]]
        .rename(columns={"label": "dx"}).drop_duplicates("md5hash"))


def family(dx):
    s = str(dx).lower()
    if "kaposi" in s:
        return "kaposi sarcoma"
    if "squamous" in s:
        return "squamous cell carcinoma"
    if "basal cell" in s:
        return "basal cell carcinoma"
    if "melanoma" in s or "lentigo maligna" in s:
        return "melanoma (all types)"
    if "mycosis" in s:
        return "mycosis fungoides"
    return "other malignant"


def prep(d):
    d = d.copy()
    d["md5hash"] = d["path"].map(lambda p: os.path.splitext(os.path.basename(p))[0])
    d = d.merge(meta, on="md5hash", how="left")
    d = d[d.y == 1].copy()
    d["fam"] = d["dx"].map(family)
    d["grp"] = np.where(d.fst <= 3, "FST1-3", np.where(d.fst <= 5, "FST4-5", "FST6"))
    d["hit"] = d.prob >= 0.5
    return d


per_seed = []
for s, d in main.items():
    p = prep(d)
    g = p.groupby(["fam", "grp"]).agg(n=("hit", "size"), sens=("hit", "mean")).reset_index()
    g["seed"] = s
    per_seed.append(g)
R = pd.concat(per_seed)
GRPS = ["FST1-3", "FST4-5", "FST6"]

print("=" * 74)
print("1. MALIGNANT IMAGES PER SUBTYPE AND SKIN GROUP (mean per seed)")
print("=" * 74)
N = R.pivot_table(index="fam", columns="grp", values="n", aggfunc="mean").reindex(columns=GRPS)
N = N.sort_values("FST6", ascending=False)
print(N.round(1).to_string())

print("\n" + "=" * 74)
print("2. SENSITIVITY PER SUBTYPE AND SKIN GROUP (mean over seeds, threshold 0.5)")
print("   Compare down each row: does the same subtype get worse at FST 6?")
print("=" * 74)
S = R.pivot_table(index="fam", columns="grp", values="sens", aggfunc="mean").reindex(columns=GRPS)
print(S.reindex(N.index).round(3).to_string())

print("\n" + "=" * 74)
print("3. STANDARDISED SENSITIVITY (same subtype mix on both sides)")
print("=" * 74)


def standardised(tab, mix_grp, target_grp):
    t = tab.pivot(index="fam", columns="grp", values="sens")
    n = tab.pivot(index="fam", columns="grp", values="n")
    if mix_grp not in n or target_grp not in t:
        return np.nan
    ok = t[target_grp].notna() & n[mix_grp].notna()
    if not ok.any():
        return np.nan
    return float((n.loc[ok, mix_grp] * t.loc[ok, target_grp]).sum() / n.loc[ok, mix_grp].sum())


def observed(tab, grp):
    t = tab[tab.grp == grp]
    return float((t.n * t.sens).sum() / t.n.sum()) if len(t) else np.nan


rows = []
for s, tab in R.groupby("seed"):
    rows.append(dict(
        seed=s,
        FST6_observed=observed(tab, "FST6"),
        FST6_at_FST1_3_mix=standardised(tab, "FST1-3", "FST6"),
        FST1_3_observed=observed(tab, "FST1-3"),
        FST1_3_at_FST6_mix=standardised(tab, "FST6", "FST1-3"),
    ))
Z = pd.DataFrame(rows)
print(Z.round(3).to_string(index=False))
print("\nmean over seeds")
print(Z.drop(columns="seed").mean().round(3).to_string())
print("""
How to read it:
  FST6_observed vs FST6_at_FST1_3_mix   if the second is much higher, FST 6 looks bad
                                        mainly because of WHICH cancers it contains.
  FST1_3_observed vs FST1_3_at_FST6_mix if the second is much lower, light-skin images
                                        also score badly on the FST 6 subtype mix.
  If FST6_at_FST1_3_mix is still far below FST1_3_observed, a skin-tone effect remains.
""")

In [ ]:
%%writefile analyze_source.py
"""
Source-site confound check: is the FST 6 cliff a skin-tone effect, or does it
come from FST 6 images being drawn from a different source site (different
photo style) than the rest?

In a Kaggle cell (same setup as analyze_fst.py):
    os.environ["OUT_DIR"] = "<folder holding probs_*.csv>"
    %run analyze_source.py
Needs the Fitzpatrick17k CSV (with its `url` column) under /kaggle/input.
CPU only, seconds. Counts for FST 6 are small, so read it as descriptive.
"""
import glob
import os
import re
from urllib.parse import urlparse

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

OUT  = os.environ.get("OUT_DIR", "/kaggle/working")
DATA = os.environ.get("DATA_ROOT", "/kaggle/input")
MIN_IMAGES = 100          # sites with fewer test images are pooled into "other"
pd.set_option("display.width", 200, "display.precision", 3)


def sens(p, y):
    m = y == 1
    return float((p[m] >= 0.5).mean()) if m.any() else np.nan


def auc(p, y):
    return float(roc_auc_score(y, p)) if len(np.unique(y)) > 1 else np.nan


main = {}
for f in glob.glob(f"{OUT}/probs_seed*.csv"):
    m = re.fullmatch(r"probs_seed(\d+)\.csv", os.path.basename(f))
    if m:
        main[int(m.group(1))] = pd.read_csv(f)
main = dict(sorted(main.items()))
if not main:
    raise SystemExit(f"No probs_seed*.csv in {OUT}")

csvs = [c for c in glob.glob(f"{DATA}/**/*.csv", recursive=True)
        if "fitz" in os.path.basename(c).lower()]
if os.environ.get("CSV_PATH"):
    csvs = [os.environ["CSV_PATH"]]
if not csvs:
    raise SystemExit(f"Fitzpatrick17k CSV not found under {DATA}")
raw = pd.read_csv(csvs[0])
if "url" not in raw.columns:
    raise SystemExit(f"No `url` column. Columns: {list(raw.columns)}")


def site_of(u):
    try:
        host = urlparse(str(u)).netloc.lower().replace("www.", "")
    except Exception:
        host = ""
    return host or "unknown"


meta = raw[["md5hash", "url"]].copy()
meta["site"] = meta["url"].map(site_of)
meta = meta[["md5hash", "site"]].drop_duplicates("md5hash")


def tag(d, keep_sites=None):
    d = d.copy()
    d["md5hash"] = d["path"].map(lambda p: os.path.splitext(os.path.basename(p))[0])
    d = d.merge(meta, on="md5hash", how="left")
    d["site"] = d["site"].fillna("unknown")
    d["grp"] = np.where(d.fst <= 3, "FST1-3", np.where(d.fst <= 5, "FST4-5", "FST6"))
    if keep_sites is not None:
        d.loc[~d.site.isin(keep_sites), "site"] = "other"
    return d


first = tag(main[min(main)])
counts = first.site.value_counts()
keep = list(counts[counts >= MIN_IMAGES].index)
print("sites with >=", MIN_IMAGES, "test images:", dict(counts[counts >= MIN_IMAGES]))

print("\n" + "=" * 74)
print("1. SOURCE-SITE MIX BY SKIN GROUP (% of that group's test images, seed 0)")
print("=" * 74)
first = tag(main[min(main)], keep)
ct = pd.crosstab(first.site, first.grp).reindex(columns=["FST1-3", "FST4-5", "FST6"])
print(ct.join((ct / ct.sum() * 100).round(1).add_suffix("_%")).to_string())
print("\nmalignant prevalence by site and skin group")
print(first.pivot_table(index="site", columns="grp", values="y", aggfunc="mean")
      .reindex(columns=["FST1-3", "FST4-5", "FST6"]).round(3).to_string())

rows = []
for s, d in main.items():
    d = tag(d, keep)
    for (site, grp), g in d.groupby(["site", "grp"]):
        p, y = g.prob.values, g.y.values
        rows.append(dict(seed=s, site=site, grp=grp, n=len(g), pos=int(y.sum()),
                         sens=sens(p, y), auc=auc(p, y)))
R = pd.DataFrame(rows)
G = ["FST1-3", "FST4-5", "FST6"]

print("\n" + "=" * 74)
print("2. SENSITIVITY AND AUC BY SITE AND SKIN GROUP (mean over seeds)")
print("   Does the FST 6 drop appear inside EACH site?")
print("=" * 74)
for col, label in (("pos", "malignant images (mean per seed)"),
                   ("sens", "sensitivity"), ("auc", "AUC")):
    print(f"\n{label}")
    print(R.pivot_table(index="site", columns="grp", values=col, aggfunc="mean")
          .reindex(columns=G).round(3).to_string())

print("\n" + "=" * 74)
print("3. WITHIN-SITE GAP: FST1-3 minus FST6 (mean over seeds)")
print("=" * 74)
P_s = R.pivot_table(index="site", columns="grp", values="sens", aggfunc="mean")
P_a = R.pivot_table(index="site", columns="grp", values="auc", aggfunc="mean")
gap = pd.DataFrame({"sens_gap": P_s["FST1-3"] - P_s["FST6"],
                    "auc_gap": P_a["FST1-3"] - P_a["FST6"]})
print(gap.round(3).to_string())
print("""
How to read it:
  A gap that is large inside every site that has FST 6 images means the cliff is
  not a source-site artefact. A gap that appears in only one site, or vanishes
  inside sites, means photo style / provenance is a competing explanation.
  Ignore rows where FST 6 has fewer than ~10 malignant images.
""")

In [ ]:
%%writefile analyze_extra.py
"""
Analyses for this notebook's runs. CPU only, about a minute.

1. EfficientNet-B0 (5 seeds): does the FST 6 cliff replicate on a second backbone?
2. Size-matched holdout: is the FST 6 recovery from darker-skin exposure or data volume?
3. Second skin-type annotation (fitzpatrick_centaur): is the cliff robust to FST label noise?

Reads probs_*.csv from OUT_DIR (default /kaggle/working) and the Fitzpatrick17k CSV
under DATA_ROOT (default /kaggle/input).
"""
import glob
import os
import re

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

OUT  = os.environ.get("OUT_DIR", "/kaggle/working")
DATA = os.environ.get("DATA_ROOT", "/kaggle/input")
NB   = int(os.environ.get("NBOOT", 1000))
rng  = np.random.default_rng(0)
pd.set_option("display.width", 200, "display.precision", 3)


def sens(p, y):
    m = y == 1
    return float((p[m] >= 0.5).mean()) if m.any() else np.nan


def spec(p, y):
    m = y == 0
    return float((p[m] < 0.5).mean()) if m.any() else np.nan


def auc(p, y):
    return float(roc_auc_score(y, p)) if len(np.unique(y)) > 1 else np.nan


def summarize(g):
    p, y = g["prob"].values, g["y"].values
    return dict(n=len(g), pos=int(y.sum()), sens=sens(p, y), spec=spec(p, y), auc=auc(p, y))


def read_seed_files(suffix):
    found = {}
    for f in glob.glob(f"{OUT}/probs_seed*{suffix}.csv"):
        m = re.fullmatch(rf"probs_seed(\d+){re.escape(suffix)}\.csv", os.path.basename(f))
        if m:
            found[int(m.group(1))] = pd.read_csv(f)
    return dict(sorted(found.items()))


def boot_gap(L, D, stat):
    pl, yl = L["prob"].values, L["y"].values
    pdk, yd = D["prob"].values, D["y"].values
    point = stat(pl, yl) - stat(pdk, yd)
    gaps = np.empty(NB)
    for b in range(NB):
        i = rng.integers(0, len(yl), len(yl))
        j = rng.integers(0, len(yd), len(yd))
        gaps[b] = stat(pl[i], yl[i]) - stat(pdk[j], yd[j])
    return point, np.nanpercentile(gaps, 2.5), np.nanpercentile(gaps, 97.5)


def per_level(runs, col="fst"):
    rows = []
    for s, d in runs.items():
        for lvl, g in d.groupby(col):
            rows.append(dict(seed=s, fst=int(lvl), **summarize(g)))
    return pd.DataFrame(rows)


def contrasts(runs, col="fst", label=""):
    spec_ = [("FST 1-3 minus FST 6", [1, 2, 3], [6]),
             ("FST 1-3 minus FST 4-5", [1, 2, 3], [4, 5]),
             ("FST 4-5 minus FST 6", [4, 5], [6])]
    for name, lo, hi in spec_:
        for metric, stat in (("sensitivity", sens), ("AUC", auc)):
            res = []
            for s, d in runs.items():
                L, D = d[d[col].isin(lo)], d[d[col].isin(hi)]
                if L.y.sum() < 5 or D.y.sum() < 5:
                    continue
                res.append(boot_gap(L, D, stat))
            if not res:
                continue
            gaps = np.array([r[0] for r in res])
            sig = sum(1 for r in res if r[1] > 0 or r[2] < 0)
            print(f"{label}{name:24s} {metric:12s} mean gap {gaps.mean():+.3f}  "
                  f"per-seed [{gaps.min():+.3f}, {gaps.max():+.3f}]  "
                  f"CI excludes 0 in {sig}/{len(res)} seeds")


main = read_seed_files("")
eff  = read_seed_files("_effb0")
hold = read_seed_files("_hold45")
hsm  = read_seed_files("_hold45_sm")
print(f"resnet50 main: {list(main)}  effb0: {list(eff)}  hold45: {list(hold)}  hold45_sm: {list(hsm)}")

# ---------------------------------------------------------------------------
print("\n" + "=" * 74)
print("1. SECOND BACKBONE: EfficientNet-B0 vs ResNet-50 (mean over seeds)")
print("=" * 74)
if eff and main:
    A = per_level(main).groupby("fst")[["sens", "auc"]].agg(["mean", "std"])
    B = per_level(eff).groupby("fst")[["sens", "auc"]].agg(["mean", "std"])
    A.columns = [f"r50_{a}_{b}" for a, b in A.columns]
    B.columns = [f"eff_{a}_{b}" for a, b in B.columns]
    print(A.join(B).round(3).to_string())
    print("\nEfficientNet-B0 contrasts")
    contrasts(eff, label="effb0  ")
    print("\nResNet-50 contrasts (for comparison)")
    contrasts(main, label="r50    ")
else:
    print("effb0 runs or resnet main runs missing - skipped")

# ---------------------------------------------------------------------------
print("\n" + "=" * 74)
print("2. SIZE-MATCHED HOLDOUT, ON FST 6 (same images for every model)")
print("   baseline     : light only, 8,851 images")
print("   +FST4,5      : light + FST 4,5, ~13,165 images (more data AND darker skin)")
print("   +FST4,5 (sm) : fewer light + FST 4,5, 8,851 images (darker skin, same volume)")
print("=" * 74)
rows = []
for s in sorted(set(hsm) | set(hold)):
    if s not in main:
        continue
    for name, src in (("baseline", main), ("+FST4,5", hold), ("+FST4,5 (sm)", hsm)):
        if s in src:
            d = src[s]
            rows.append(dict(seed=s, subset="FST6", model=name, **summarize(d[d.fst == 6])))
            rows.append(dict(seed=s, subset="light test", model=name,
                             **summarize(d[d.group == "light"])))
if rows:
    H = pd.DataFrame(rows)
    print(H.round(3).to_string(index=False))
    print("\nmean over seeds")
    print(H.groupby(["subset", "model"], sort=False)[["sens", "spec", "auc"]].mean().round(3).to_string())
    print("""
How to read it:
  +FST4,5 (sm) close to +FST4,5, both above baseline -> darker-skin exposure helps,
                                                      not just extra volume.
  +FST4,5 (sm) close to baseline                     -> most of the gain was volume.
""")
else:
    print("no holdout runs found - skipped")

# ---------------------------------------------------------------------------
print("=" * 74)
print("3. SECOND SKIN-TYPE ANNOTATION (fitzpatrick_centaur)")
print("   Caveat: training used fitzpatrick_scale. Grouping test images by the other")
print("   annotation is a robustness check on the test side, not a clean re-run.")
print("=" * 74)
csvs = [c for c in glob.glob(f"{DATA}/**/*.csv", recursive=True)
        if "fitz" in os.path.basename(c).lower()]
if not csvs:
    print("Fitzpatrick17k CSV not found - skipped")
else:
    raw = pd.read_csv(csvs[0])
    if "fitzpatrick_centaur" not in raw.columns:
        print("no fitzpatrick_centaur column - skipped")
    else:
        meta = raw[["md5hash", "fitzpatrick_scale", "fitzpatrick_centaur"]].drop_duplicates("md5hash")
        meta["fst_c"] = pd.to_numeric(meta["fitzpatrick_centaur"], errors="coerce")
        meta.loc[~meta.fst_c.between(1, 6), "fst_c"] = np.nan

        def add_c(d):
            d = d.copy()
            d["md5hash"] = d["path"].map(lambda p: os.path.splitext(os.path.basename(p))[0])
            d = d.merge(meta[["md5hash", "fst_c"]], on="md5hash", how="left")
            return d

        cm = {s: add_c(d) for s, d in main.items()}
        d0 = cm[min(cm)].dropna(subset=["fst_c"])
        print(f"test images with a valid second label (seed {min(cm)}): {len(d0)} "
              f"of {len(cm[min(cm)])}")
        ag = (d0.fst == d0.fst_c).mean()
        w1 = ((d0.fst - d0.fst_c).abs() <= 1).mean()
        print(f"exact agreement {ag:.1%}   within one level {w1:.1%}")
        print("\nrows = fitzpatrick_scale (used for training), cols = fitzpatrick_centaur")
        print(pd.crosstab(d0.fst.astype(int), d0.fst_c.astype(int)).to_string())

        print("\n(a) grouped by the SECOND annotation (mean over seeds)")
        runs_c = {s: d.dropna(subset=["fst_c"]).assign(fst_c=lambda x: x.fst_c.astype(int))
                  for s, d in cm.items()}
        Tc = per_level(runs_c, col="fst_c")
        print(Tc.groupby("fst")[["n", "pos", "sens", "auc"]].mean().round(3).to_string())
        contrasts(runs_c, col="fst_c", label="centaur ")

        print("\n(b) only images where BOTH annotators agree (mean over seeds)")
        runs_a = {s: d[d.fst == d.fst_c] for s, d in runs_c.items()}
        Ta = per_level(runs_a)
        print(Ta.groupby("fst")[["n", "pos", "sens", "auc"]].mean().round(3).to_string())
        contrasts(runs_a, label="agreed  ")
        print("""
How to read it:
  If FST 6 is still far below FST 1-3 under the second annotation and on the
  agreed subset, the cliff is not an artefact of one annotator's skin-type labels.
  Agreed-subset counts can be small; check the pos column before trusting a row.
""")

In [ ]:
%%writefile make_figures.py
"""
Paper figures at IEEE width (column 3.5 in, page 7.16 in), as vector PDF + 300 dpi PNG.
Fonts are embedded as TrueType (pdf.fonttype 42), which IEEE PDF checks require.

In a Kaggle cell:
    os.environ["OUT_DIR"] = "<folder holding probs_*.csv>"
    %run make_figures.py
Writes to FIG_DIR (default /kaggle/working/figs). Download that folder.

fig1_per_fst        sensitivity and AUC by Fitzpatrick level, mean +/- SD, dots = seeds
fig2_subtype        sensitivity by cancer subtype and skin group (needs the CSV; descriptive)
fig3_recovery       FST 6 recovery: baseline vs +FST 4,5 (holdout) vs oversample FST 6,
                    each compared on exactly the same test images
"""
import glob
import os
import re

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

OUT  = os.environ.get("OUT_DIR", "/kaggle/working")
DATA = os.environ.get("DATA_ROOT", "/kaggle/input")
FIG  = os.environ.get("FIG_DIR", "/kaggle/working/figs")
os.makedirs(FIG, exist_ok=True)

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "Nimbus Roman", "DejaVu Serif"],
    "font.size": 7.5, "axes.labelsize": 7.5, "axes.titlesize": 8,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 6.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "pdf.fonttype": 42, "ps.fonttype": 42, "savefig.dpi": 300,
})
BASE, HI, REF = "#7A9CC6", "#D55E00", "#333333"      # FST 6 gets colour AND hatching
GRP_COL = {"FST1-3": "#7A9CC6", "FST4-5": "#E69F00", "FST6": "#D55E00"}
ROMAN = {1: "I", 2: "II", 3: "III", 4: "IV", 5: "V", 6: "VI"}


def sens(p, y):
    m = y == 1
    return float((p[m] >= 0.5).mean()) if m.any() else np.nan


def auc(p, y):
    return float(roc_auc_score(y, p)) if len(np.unique(y)) > 1 else np.nan


def read(suffix):
    found = {}
    for f in glob.glob(f"{OUT}/probs_seed*{suffix}.csv"):
        m = re.fullmatch(rf"probs_seed(\d+){re.escape(suffix)}\.csv", os.path.basename(f))
        if m:
            found[int(m.group(1))] = pd.read_csv(f)
    return dict(sorted(found.items()))


main, mit, hold = read(""), read("_mitigate6"), read("_hold45")
hold_sm = read("_hold45_sm")
if not main:
    raise SystemExit(f"No probs_seed*.csv in {OUT}")


def save(fig, name):
    fig.savefig(f"{FIG}/{name}.pdf", bbox_inches="tight")
    fig.savefig(f"{FIG}/{name}.png", bbox_inches="tight")
    plt.close(fig)
    print("wrote", f"{FIG}/{name}.pdf / .png")


def dots(ax, x, vals, color="black"):
    vals = np.asarray(vals, float)
    off = np.linspace(-0.14, 0.14, len(vals)) if len(vals) > 1 else np.zeros(1)
    ax.scatter(x + off, vals, s=5, color=color, zorder=3, linewidths=0)


# ---------------------------------------------------------------- Figure 1
rows = []
for s, d in main.items():
    for lvl, g in d.groupby("fst"):
        p, y = g.prob.values, g.y.values
        rows.append(dict(seed=s, fst=int(lvl), sens=sens(p, y), auc=auc(p, y)))
    L = d[d.fst <= 3]
    rows.append(dict(seed=s, fst=0, sens=sens(L.prob.values, L.y.values),
                     auc=auc(L.prob.values, L.y.values)))      # fst 0 = pooled FST 1-3
T = pd.DataFrame(rows)
levels = [l for l in sorted(T.fst.unique()) if l > 0]

fig, axes = plt.subplots(1, 2, figsize=(7.16, 2.3))
for ax, col, ylab in ((axes[0], "sens", "Sensitivity"), (axes[1], "auc", "AUC")):
    for i, l in enumerate(levels):
        v = T[T.fst == l][col].values
        ax.bar(i, v.mean(), yerr=v.std(ddof=1) if len(v) > 1 else 0, capsize=2,
               color=HI if l == 6 else BASE, hatch="///" if l == 6 else None,
               edgecolor="black", linewidth=0.5, error_kw=dict(lw=0.6))
        dots(ax, i, v)
    ref = T[T.fst == 0][col].mean()
    ax.axhline(ref, ls="--", lw=0.7, color=REF)
    ax.text(len(levels) - 0.5, ref + 0.02, "FST I–III pooled", ha="right", va="bottom", fontsize=6.5)
    ax.set_xticks(range(len(levels)))
    ax.set_xticklabels([ROMAN[l] for l in levels])
    ax.set_xlabel("Fitzpatrick skin type")
    ax.set_ylabel(ylab)
    ax.set_ylim(0, 1)
axes[0].set_title("(a) Sensitivity at threshold 0.5", loc="left")
axes[1].set_title("(b) AUC", loc="left")
fig.tight_layout()
save(fig, "fig1_per_fst")

# ---------------------------------------------------------------- Figure 2
csvs = [c for c in glob.glob(f"{DATA}/**/*.csv", recursive=True)
        if "fitz" in os.path.basename(c).lower()]
if os.environ.get("CSV_PATH"):
    csvs = [os.environ["CSV_PATH"]]
if csvs:
    meta = (pd.read_csv(csvs[0])[["md5hash", "label"]]
            .rename(columns={"label": "dx"}).drop_duplicates("md5hash"))

    def family(dx):
        s = str(dx).lower()
        if "kaposi" in s:
            return "Kaposi\nsarcoma"
        if "squamous" in s:
            return "Squamous\ncell"
        if "basal cell" in s:
            return "Basal\ncell"
        if "melanoma" in s or "lentigo maligna" in s:
            return "Melanoma"
        return None

    per = []
    for s, d in main.items():
        d = d.copy()
        d["md5hash"] = d["path"].map(lambda p: os.path.splitext(os.path.basename(p))[0])
        d = d.merge(meta, on="md5hash", how="left")
        d = d[d.y == 1]
        d["fam"] = d["dx"].map(family)
        d = d.dropna(subset=["fam"])
        d["grp"] = np.where(d.fst <= 3, "FST1-3", np.where(d.fst <= 5, "FST4-5", "FST6"))
        d["hit"] = d.prob >= 0.5
        g = d.groupby(["fam", "grp"]).agg(n=("hit", "size"), sens=("hit", "mean")).reset_index()
        g["seed"] = s
        per.append(g)
    R = pd.concat(per)
    S = R.pivot_table(index="fam", columns="grp", values="sens", aggfunc="mean")
    N = R.pivot_table(index="fam", columns="grp", values="n", aggfunc="mean")
    order = [f for f in N.sort_values("FST6", ascending=False).index if N.loc[f, "FST6"] >= 5]
    fig, ax = plt.subplots(figsize=(3.5, 2.3))
    w = 0.26
    for j, grp in enumerate(["FST1-3", "FST4-5", "FST6"]):
        xs = np.arange(len(order)) + (j - 1) * w
        ax.bar(xs, [S.loc[f, grp] for f in order], w, color=GRP_COL[grp],
               hatch="///" if grp == "FST6" else None, edgecolor="black", linewidth=0.5,
               label={"FST1-3": "FST I–III", "FST4-5": "FST IV–V", "FST6": "FST VI"}[grp])
    for i, f in enumerate(order):
        ax.text(i + w, S.loc[f, "FST6"] + 0.02, f"n={int(round(N.loc[f, 'FST6']))}",
                ha="center", va="bottom", fontsize=5.5)
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels(order, fontsize=6.5)
    ax.set_ylabel("Sensitivity")
    ax.set_ylim(0, 1)
    ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.18))
    fig.tight_layout()
    save(fig, "fig2_subtype")
else:
    print("Fitzpatrick17k CSV not found - skipped fig2_subtype")

# ---------------------------------------------------------------- Figure 3
pairs = []     # (label, [per-seed sens], [per-seed auc], colour, hatch)
if hold or mit:
    def collect(src, subset_fn):
        out = []
        for s, m in src.items():
            if s not in main:
                continue
            b, t = subset_fn(main[s], m)
            out.append((sens(b.prob.values, b.y.values), auc(b.prob.values, b.y.values),
                        sens(t.prob.values, t.y.values), auc(t.prob.values, t.y.values)))
        return np.array(out)

    def hold_fn(b, h):
        return b[b.fst == 6], h[h.fst == 6]

    def mit_fn(b, m):
        m6 = m[m.fst == 6]
        return b[(b.fst == 6) & b.path.isin(m6.path)], m6[m6.path.isin(b.path)]

    xs, labels, sv, av, cols, centres = [], [], [], [], [], []
    x = 0.0
    groups = (("all FST VI images", hold_fn,
               (("+ IV–V\nmore data", hold), ("+ IV–V\nsame size", hold_sm))),
              ("held-out half of FST VI", mit_fn, (("oversample\nVI", mit),)))
    for name, fn, treats in groups:
        arms = [(t, collect(src, fn)) for t, src in treats if src]
        arms = [(t, r) for t, r in arms if len(r)]
        if not arms:
            continue
        # baseline uses the seeds of the first arm; every arm is tested on the same images
        r0 = arms[0][1]
        bars = [("baseline", r0[:, 0], r0[:, 1], BASE)]
        for t, r in arms:
            bars.append((t, r[:, 2], r[:, 3], HI))
        for k, (lab, sv_, av_, c) in enumerate(bars):
            xs.append(x + k); labels.append(lab); sv.append(sv_); av.append(av_); cols.append(c)
        centres.append((x + (len(bars) - 1) / 2, "tested on\n" + name))
        x += len(bars) + 0.8
    ref_s = np.mean([sens(d[d.fst <= 3].prob.values, d[d.fst <= 3].y.values) for d in main.values()])
    ref_a = np.mean([auc(d[d.fst <= 3].prob.values, d[d.fst <= 3].y.values) for d in main.values()])
    fig, axes = plt.subplots(1, 2, figsize=(7.16, 2.4))
    for ax, vals, ref, ylab in ((axes[0], sv, ref_s, "FST VI sensitivity"), (axes[1], av, ref_a, "FST VI AUC")):
        for xi, v, c in zip(xs, vals, cols):
            ax.bar(xi, v.mean(), 0.8, yerr=v.std(ddof=1) if len(v) > 1 else 0, capsize=2, color=c,
                   hatch="///" if c == HI else None, edgecolor="black", linewidth=0.5,
                   error_kw=dict(lw=0.6))
            dots(ax, xi, v)
        ax.axhline(ref, ls="--", lw=0.7, color=REF)
        ax.text(xs[-1] + 0.5, ref + 0.02, "FST I–III", ha="right", va="bottom", fontsize=6.5)
        ax.set_xticks(xs)
        ax.set_xticklabels(labels, fontsize=6)
        for cx, cap in centres:
            ax.annotate(cap, xy=(cx, 0), xycoords=("data", "axes fraction"),
                        xytext=(0, -30), textcoords="offset points",
                        ha="center", va="top", fontsize=6.5, style="italic", linespacing=1.15)
        ax.set_ylabel(ylab)
        ax.set_ylim(0, 1)
    axes[0].set_title("(a) Sensitivity", loc="left")
    axes[1].set_title("(b) AUC", loc="left")
    fig.tight_layout()
    save(fig, "fig3_recovery")
else:
    print("no holdout/mitigation files - skipped fig3_recovery")

## A. EfficientNet-B0 (5 seeds)

In [ ]:
import os
os.environ.update(SMOKE="0", EVAL_ONLY="0", EPOCHS="10", MITIGATE="0", HOLDOUT_FST="", SIZE_MATCH="0", ARCH="effb0")
for s in range(5):
    if os.path.exists(f"/kaggle/working/probs_seed{s}_effb0.csv"):
        print(f"seed {s} effb0 already done - skipped"); continue
    os.environ["SEED"] = str(s)
    print(f"\n{'='*70}\nEFFICIENTNET-B0 (ORIGINAL IMAGES) - SEED {s}\n{'='*70}", flush=True)
    %run skin_tone_gap.py

## B. Holdout: +FST IV–V in training (ResNet-50, 3 seeds)

In [ ]:
import os
os.environ.update(SMOKE="0", EVAL_ONLY="0", EPOCHS="10", MITIGATE="0", HOLDOUT_FST="4,5", SIZE_MATCH="0", ARCH="resnet50")
for s in range(3):
    if os.path.exists(f"/kaggle/working/probs_seed{s}_hold45.csv"):
        print(f"seed {s} hold45 already done - skipped"); continue
    os.environ["SEED"] = str(s)
    print(f"\n{'='*70}\nHOLDOUT +FST IV-V (ORIGINAL IMAGES) - SEED {s}\n{'='*70}", flush=True)
    %run skin_tone_gap.py

## C. Size-matched holdout (ResNet-50, 3 seeds)

In [ ]:
import os
os.environ.update(SMOKE="0", EVAL_ONLY="0", EPOCHS="10", MITIGATE="0", HOLDOUT_FST="4,5", SIZE_MATCH="1", ARCH="resnet50")
for s in range(3):
    if os.path.exists(f"/kaggle/working/probs_seed{s}_hold45_sm.csv"):
        print(f"seed {s} hold45_sm already done - skipped"); continue
    os.environ["SEED"] = str(s)
    print(f"\n{'='*70}\nSIZE-MATCHED HOLDOUT (ORIGINAL IMAGES) - SEED {s}\n{'='*70}", flush=True)
    %run skin_tone_gap.py
os.environ.update(HOLDOUT_FST="", SIZE_MATCH="0")

## D. Analyses and figures (CPU)

In [ ]:
%run analyze_fst.py

In [ ]:
%run analyze_extra.py

In [ ]:
%run make_figures.py